In [1]:
# CELL 1: Connect to Google Drive
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
print("✅ Google Drive connected successfully!")
print("Your files are now accessible at: /content/drive/MyDrive/")

# CELL 2: Unzip hw3.zip file
import zipfile
import os

# Path to your hw3.zip file in Google Drive
zip_path = '/content/drive/MyDrive/hw3.zip'

# Where to extract the files (in Colab's local storage)
extract_path = '/content/hw3/'

# Check if zip file exists
if os.path.exists(zip_path):
    print(f"Found hw3.zip at: {zip_path}")

    # Create extraction directory
    os.makedirs(extract_path, exist_ok=True)

    # Extract the zip file
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print(f"✅ Successfully extracted hw3.zip to: {extract_path}")

    # Show extracted contents
    print("\nExtracted files and folders:")
    for root, dirs, files in os.walk(extract_path):
        level = root.replace(extract_path, '').count(os.sep)
        indent = '  ' * level
        folder_name = os.path.basename(root) or 'hw3'
        print(f"{indent}{folder_name}/")

        subindent = '  ' * (level + 1)
        for file in files:
            print(f"{subindent}{file}")

else:
    print(f"❌ hw3.zip not found at: {zip_path}")
    print("Please make sure:")
    print("1. The file is uploaded to your Google Drive")
    print("2. The file is named exactly 'hw3.zip'")
    print("3. The file is in the root of your Drive (MyDrive folder)")

    # List zip files in Drive to help debug
    drive_path = '/content/drive/MyDrive/'
    print(f"\nZip files found in {drive_path}:")
    for file in os.listdir(drive_path):
        if file.endswith('.zip'):
            print(f"  - {file}")

Streaming output truncated to the last 5000 lines.
              002.jpeg
              003.jpeg
            1629/
              002.jpeg
              001.jpeg
            1699/
              002.jpeg
              001.jpeg
            978/
              002.jpeg
              003.jpeg
              001.jpeg
            610/
              002.jpeg
              003.jpeg
              001.jpeg
            209/
              001.jpeg
            555/
              003.jpeg
              001.jpeg
            321/
              002.jpeg
              003.jpeg
              001.jpeg
            412/
              002.jpeg
              001.jpeg
            1616/
              002.jpeg
              001.jpeg
            774/
              002.jpeg
              003.jpeg
              001.jpeg
            1370/
              003.jpeg
            183/
              002.jpeg
            1445/
              002.jpeg
              003.jpeg
              001.jpeg
            1247/
              0

In [2]:
import json

json_file_path = '/content/hw3/hw3/json_1/phase_2/0000_all_img_link_added.json'

try:
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Overview of the file
    print("### Overview of the JSON file:")
    print(f"- Total number of entries: {len(data)}")
    if data:
        print("- Structure of the first entry:")
        print(json.dumps(data[0], indent=4))
    else:
        print("- The file is empty.")

    # List of people, iteration numbers, and image links
    print("\n### People, Iteration Numbers, and Image Links:")
    for item in data:
        iteration = item.get("iteration", "N/A")
        text = item.get("textual_content_response", "No text content")
        image_links = item.get("image_data", {}).get("image_links", [])

        print(f"\nIteration: {iteration}")
        print(f"Text Content: {text}")
        if image_links:
            print("Image Links:")
            for i, img_info in enumerate(image_links):
                print(f"  - Image {i+1}: {img_info.get('url', 'N/A')}")
        else:
            print("  No image links found.")

except FileNotFoundError:
    print(f"Error: The file was not found at {json_file_path}")
except json.JSONDecodeError:
    print(f"Error: Could not decode JSON from the file at {json_file_path}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

### Overview of the JSON file:
- Total number of entries: 959
- Structure of the first entry:
{
    "iteration": 1,
    "json_pure": {
        "name": "\u06a9\u0627\u0645\u0628\u06cc\u0632 \u0622\u062a\u0627\u0628\u0627\u06cc",
        "sex": "male",
        "nick-names": [],
        "birth": {
            "date": "1939-02-02",
            "location": {
                "province": "\u062a\u0647\u0631\u0627\u0646",
                "city": "\u062a\u0647\u0631\u0627\u0646",
                "coordinates": {
                    "latitude": "35.6895",
                    "longitude": "51.3890"
                }
            }
        },
        "death": {
            "date": "",
            "location": {
                "province": "",
                "city": "",
                "coordinates": {
                    "latitude": "",
                    "longitude": ""
                }
            },
            "tomb_location": {
                "province": "",
                "city": "",
    

In [3]:
import json
import pandas as pd
import os
from pathlib import Path

def process_single_json_item(data, all_qa_data, source_name, item_index=0):
    """
    Process a single JSON item (dictionary) and extract Q&A data
    """
    try:
        # Extract basic information
        iteration = data.get('iteration', '')

        # Try different locations for name
        name = ''
        if 'json_pure' in data and isinstance(data['json_pure'], dict):
            name = data['json_pure'].get('name', '')
        elif 'json_refined' in data and isinstance(data['json_refined'], dict):
            name = data['json_refined'].get('name', '')
        elif 'name' in data:
            name = data.get('name', '')

        print(f"  - Item {item_index + 1}: Name: {name}")
        print(f"  - Item {item_index + 1}: Iteration: {iteration}")

        # Extract questions and answers
        qa_response = data.get('question_and_answer_response', {})
        if isinstance(qa_response, dict):
            questions_answers = qa_response.get('questions_and_answers', [])
        else:
            questions_answers = []

        print(f"  - Item {item_index + 1}: Found {len(questions_answers)} Q&A pairs")

        # Add each Q&A pair to our data list
        for qa in questions_answers:
            if isinstance(qa, dict):
                all_qa_data.append({
                    'question': qa.get('question', ''),
                    'answer': qa.get('answer', ''),
                    'name': name,
                    'iteration': iteration
                })

        return len(questions_answers)

    except Exception as e:
        print(f"  - Error processing item {item_index + 1} from {source_name}: {e}")
        return 0

def extract_qa_from_json_files():
    """
    Extract question-answer data from JSON files and create a CSV
    """
    # Define the base path and JSON file locations
    base_path = '/content/hw3/hw3'
    json_folders = ['json_1', 'json_3', 'json_4']

    # List to store all Q&A data
    all_qa_data = []

    # Process each JSON folder
    for folder in json_folders:
        json_file_path = os.path.join(base_path, folder, 'phase_2', '0000_all_img_link_added.json')

        print(f"\nProcessing: {json_file_path}")

        # Check if file exists
        if not os.path.exists(json_file_path):
            print(f"❌ File not found: {json_file_path}")
            continue

        try:
            # Read JSON file with error handling for large files
            print("  - Reading file...")

            # Try to read the entire file first
            try:
                with open(json_file_path, 'r', encoding='utf-8') as file:
                    data = json.load(file)

                print(f"  - Successfully loaded JSON with {len(data)} items")

            except json.JSONDecodeError as e:
                print(f"  - JSON decode error: {e}")
                print("  - Trying alternative parsing methods...")

                # Try to fix common issues and read again
                with open(json_file_path, 'r', encoding='utf-8') as file:
                    content = file.read()

                # Try to fix trailing comma or bracket issues
                content = content.strip()
                if content.endswith(','):
                    content = content[:-1]
                if not content.endswith(']'):
                    content += ']'

                data = json.loads(content)
                print(f"  - Fixed and loaded JSON with {len(data)} items")

            # Process the data (should be a list)
            if isinstance(data, list):
                print(f"  - Processing {len(data)} items from {folder}")

                for i, item in enumerate(data):
                    if isinstance(item, dict):
                        process_single_json_item(item, all_qa_data, folder, i)
                    else:
                        print(f"  - Item {i + 1} is not a dictionary, skipping")

            else:
                print(f"  - Data is not a list (type: {type(data)}), trying to process as single item")
                process_single_json_item(data, all_qa_data, folder, 0)

        except Exception as e:
            print(f"❌ Error processing {json_file_path}: {e}")

            # Show first few characters for debugging
            try:
                with open(json_file_path, 'r', encoding='utf-8') as file:
                    first_chars = file.read(500)
                    print(f"  - First 500 characters: {first_chars}...")
            except:
                pass

    # Create DataFrame
    if all_qa_data:
        df = pd.DataFrame(all_qa_data)

        # Reorder columns as requested: question, answer, name, iteration
        df = df[['question', 'answer', 'name', 'iteration']]

        # Save to CSV
        output_file = '/content/questions_answers.csv'
        df.to_csv(output_file, index=False, encoding='utf-8')

        print(f"\n✅ CSV file created successfully!")
        print(f"📁 Output file: {output_file}")
        print(f"📊 Total records: {len(all_qa_data)}")

        # Show preview of the data
        print("\n📋 Preview of the CSV data:")
        print(df.head(10))

        # Show summary by person and iteration with error handling
        print("\n📈 Summary by person:")
        try:
            # First, let's check for any remaining problematic data
            print("  - Checking data types...")
            print(f"  - Name column type: {df['name'].dtype}")
            print(f"  - Iteration column type: {df['iteration'].dtype}")

            # Try to identify any remaining list values
            name_types = df['name'].apply(type).unique()
            iteration_types = df['iteration'].apply(type).unique()

            print(f"  - Name value types: {name_types}")
            print(f"  - Iteration value types: {iteration_types}")

            # Clean any remaining non-string values
            df['name'] = df['name'].astype(str)
            df['iteration'] = df['iteration'].astype(str)

            summary = df.groupby(['name', 'iteration']).size().reset_index(name='question_count')
            print("  - Top 10 people with most questions:")
            print(summary.sort_values('question_count', ascending=False).head(10))

        except Exception as e:
            print(f"  - Error creating summary: {e}")
            print("  - Showing basic info instead:")
            try:
                print(f"  - Unique names: {df['name'].nunique()}")
                print(f"  - Unique iterations: {df['iteration'].nunique()}")
                print(f"  - Sample names: {df['name'].head().tolist()}")
                print(f"  - Sample iterations: {df['iteration'].head().tolist()}")
            except Exception as e2:
                print(f"  - Even basic info failed: {e2}")
                print("  - But CSV file was created successfully!")

        # Show total by folder (estimated)
        print(f"\n📁 Estimated breakdown:")
        for folder in json_folders:
            folder_data = [item for item in all_qa_data if folder in str(item.get('iteration', ''))]
            print(f"  - {folder}: ~{len(folder_data)} questions")

        return df
    else:
        print("❌ No data found to create CSV")
        return None

# Run the extraction
print("🚀 Starting extraction process...")
df = extract_qa_from_json_files()

if df is not None:
    print(f"\n🎉 Process completed! Check /content/questions_answers.csv")
    print(f"📊 Final CSV shape: {df.shape}")
else:
    print("\n❌ Process failed - no data extracted")

Streaming output truncated to the last 5000 lines.
  - Item 683: Found 5 Q&A pairs
  - Item 684: Name: یوسف صیادی
  - Item 684: Iteration: 685
  - Item 684: Found 5 Q&A pairs
  - Item 685: Name: مهدی فخیم‌زاده
  - Item 685: Iteration: 686
  - Item 685: Found 5 Q&A pairs
  - Item 686: Name: رضا فیض نوروزی
  - Item 686: Iteration: 687
  - Item 686: Found 5 Q&A pairs
  - Item 687: Name: خاطره حاتمی
  - Item 687: Iteration: 688
  - Item 687: Found 5 Q&A pairs
  - Item 688: Name: غلامحسین نقشینه
  - Item 688: Iteration: 689
  - Item 688: Found 5 Q&A pairs
  - Item 689: Name: فرحناز منافی ظاهر
  - Item 689: Iteration: 690
  - Item 689: Found 5 Q&A pairs
  - Item 690: Name: مریم امیرجلالی
  - Item 690: Iteration: 691
  - Item 690: Found 5 Q&A pairs
  - Item 691: Name: معصومه اسکندری
  - Item 691: Iteration: 692
  - Item 691: Found 5 Q&A pairs
  - Item 692: Name: ایرن زازیانس
  - Item 692: Iteration: 693
  - Item 692: Found 5 Q&A pairs
  - Item 693: Name: مارکو گریگوریان
  - Item 693: Iteratio

In [4]:
import pandas as pd
import numpy as np

# Load the CSV file
df = pd.read_csv('/content/questions_answers.csv')

print(f"Original data shape: {df.shape}")
print("First 5 rows before shuffling:")
print(df.head())

# Shuffle the data
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save the shuffled data
df_shuffled.to_csv('/content/questions_answers_shuffled.csv', index=False, encoding='utf-8')

print(f"\nShuffled data shape: {df_shuffled.shape}")
print("First 5 rows after shuffling:")
print(df_shuffled.head())

print(f"\nShuffled CSV saved as: /content/questions_answers_shuffled.csv")

Original data shape: (16415, 4)
First 5 rows before shuffling:
                                            question  \
0      کامبیز آتابای در چه تاریخی در تهران متولد شد؟   
1  کامبیز آتابای در کدام دوران‌های سیاسی ایران فع...   
2  کامبیز آتابای چه سمت‌هایی در دربار پهلوی داشته...   
3   کامبیز آتابای در فوتبال ایران چه سمتی داشته است؟   
4    کامبیز آتابای در فوتبال آسیا چه سمتی داشته است؟   

                                              answer           name  iteration  
0  کامبیز آتابای در تاریخ ۱۳۱۷/۱۱/۱۳ در تهران متو...  کامبیز آتابای          1  
1  او در دوران دودمان پهلوی و جمهوری اسلامی ایران...  کامبیز آتابای          1  
2  او سابقه فعالیت به عنوان مدیر کل فنی و خدمات ع...  کامبیز آتابای          1  
3  کامبیز آتابای دهمین رئیس فدراسیون فوتبال ایران...  کامبیز آتابای          1  
4     او ششمین رئیس کنفدراسیون فوتبال آسیا بوده است.  کامبیز آتابای          1  

Shuffled data shape: (16415, 4)
First 5 rows after shuffling:
                                            questio

In [5]:
import os
import shutil
from pathlib import Path

def collect_all_image_folders():
    """
    Collect all subfolders from json_1, json_3, json_4 image directories into one central folder
    """
    # Base paths
    base_path = '/content/hw3/hw3'
    json_folders = ['json_1', 'json_3', 'json_4']

    # Create central folder for all image subfolders
    central_folder = '/content/hw3/all_image_folders2'
    os.makedirs(central_folder, exist_ok=True)

    print(f"Central folder created: {central_folder}")

    total_folders = 0
    copied_folders = 0

    # Process each JSON folder
    for folder in json_folders:
        images_path = os.path.join(base_path, folder, 'phase_2', 'downloaded_images_filter')

        print(f"\nProcessing: {folder}")
        print(f"Looking in: {images_path}")

        if not os.path.exists(images_path):
            print(f"  - Images folder not found: {images_path}")
            continue

        # Get all subfolders in the images directory
        try:
            items = os.listdir(images_path)
            subfolders = [item for item in items if os.path.isdir(os.path.join(images_path, item))]

            print(f"  - Found {len(subfolders)} subfolders: {subfolders[:5]}{'...' if len(subfolders) > 5 else ''}")
            total_folders += len(subfolders)

            # Copy each subfolder with a prefix to avoid name conflicts
            for subfolder in subfolders:
                source_folder_path = os.path.join(images_path, subfolder)

                # Create new folder name with source prefix to avoid conflicts
                new_folder_name = f"{folder}_{subfolder}"
                destination_folder_path = os.path.join(central_folder, new_folder_name)

                try:
                    # Copy entire folder with all its contents
                    shutil.copytree(source_folder_path, destination_folder_path)
                    copied_folders += 1

                    # Count images in this folder
                    image_count = count_images_in_folder(destination_folder_path)
                    print(f"    - Copied '{subfolder}' -> '{new_folder_name}' ({image_count} images)")

                except Exception as e:
                    print(f"    - Error copying {subfolder}: {e}")

        except Exception as e:
            print(f"  - Error processing {folder}: {e}")

    print(f"\n✅ Collection complete!")
    print(f"📁 Central folder: {central_folder}")
    print(f"📊 Total subfolders found: {total_folders}")
    print(f"📋 Successfully copied: {copied_folders}")

    # Show final summary
    try:
        final_folders = [f for f in os.listdir(central_folder) if os.path.isdir(os.path.join(central_folder, f))]
        total_images = 0

        print(f"📈 Final count in central folder: {len(final_folders)} subfolders")

        # Show breakdown by source folder and count total images
        print(f"\n📋 Breakdown by source:")
        for folder in json_folders:
            source_folders = [f for f in final_folders if f.startswith(f"{folder}_")]
            folder_images = 0
            for sf in source_folders:
                folder_images += count_images_in_folder(os.path.join(central_folder, sf))
            total_images += folder_images
            print(f"  - {folder}: {len(source_folders)} folders, {folder_images} images")

        print(f"🎯 Total images across all folders: {total_images}")

    except Exception as e:
        print(f"Error getting final summary: {e}")

    return central_folder, copied_folders

def count_images_in_folder(folder_path):
    """
    Count image files in a folder and its subfolders
    """
    image_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tiff', '.svg')
    count = 0

    try:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if file.lower().endswith(image_extensions):
                    count += 1
    except:
        pass

    return count

def explore_folder_structure():
    """
    Explore and show the structure of the image folders to understand what's there
    """
    base_path = '/content/hw3/hw3'
    json_folders = ['json_1', 'json_3', 'json_4']

    print("🔍 Exploring folder structure:")

    for folder in json_folders:
        images_path = os.path.join(base_path, folder, 'phase_2', 'downloaded_images_filter')

        print(f"\n📁 {folder}:")
        print(f"   Path: {images_path}")

        if os.path.exists(images_path):
            try:
                items = os.listdir(images_path)
                folders = [item for item in items if os.path.isdir(os.path.join(images_path, item))]
                files = [item for item in items if os.path.isfile(os.path.join(images_path, item))]

                print(f"   - Subfolders: {len(folders)}")
                print(f"   - Files: {len(files)}")

                if folders:
                    print(f"   - Sample subfolders: {folders[:3]}")

                    # Check contents of first subfolder
                    sample_folder = os.path.join(images_path, folders[0])
                    sample_contents = os.listdir(sample_folder)
                    sample_images = [f for f in sample_contents if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp'))]
                    print(f"   - Sample folder '{folders[0]}' contains {len(sample_images)} images")

            except Exception as e:
                print(f"   - Error exploring: {e}")
        else:
            print(f"   - Path does not exist")

# First explore the structure
print("🔍 First, let's explore the folder structure:")
explore_folder_structure()

print("\n" + "="*50)
print("🚀 Starting folder collection...")

# Then collect all folders
central_folder, count = collect_all_image_folders()

print(f"\n🎉 All image folders collected in: {central_folder}")
print(f"📊 Total folders copied: {count}")

🔍 First, let's explore the folder structure:
🔍 Exploring folder structure:

📁 json_1:
   Path: /content/hw3/hw3/json_1/phase_2/downloaded_images_filter
   - Subfolders: 932
   - Files: 0
   - Sample subfolders: ['446', '306', '741']
   - Sample folder '446' contains 2 images

📁 json_3:
   Path: /content/hw3/hw3/json_3/phase_2/downloaded_images_filter
   - Subfolders: 1639
   - Files: 0
   - Sample subfolders: ['446', '306', '1449']
   - Sample folder '446' contains 3 images

📁 json_4:
   Path: /content/hw3/hw3/json_4/phase_2/downloaded_images_filter
   - Subfolders: 560
   - Files: 0
   - Sample subfolders: ['446', '306', '104']
   - Sample folder '446' contains 1 images

🚀 Starting folder collection...
Central folder created: /content/hw3/all_image_folders2

Processing: json_1
Looking in: /content/hw3/hw3/json_1/phase_2/downloaded_images_filter
  - Found 932 subfolders: ['446', '306', '741', '104', '501']...
    - Copied '446' -> 'json_1_446' (2 images)
    - Copied '306' -> 'json_1_3

In [6]:
import os

# Define the parent directory
parent_dir = '/content/hw3/all_image_folders2'

# Iterate through all items in the parent directory
for item in os.listdir(parent_dir):
    item_path = os.path.join(parent_dir, item)
    if os.path.isdir(item_path) and item.startswith('json_'):
        # Extract the last number after the last underscore
        # Split by '_' and take the last part, assuming it's a number
        parts = item.split('_')
        if len(parts) > 1:
            last_part = parts[-1]
            # Check if last_part is a number
            if last_part.isdigit():
                new_name = last_part
                new_path = os.path.join(parent_dir, new_name)

                # Check if the new name already exists
                if os.path.exists(new_path):
                    print(f"Skipping {item} because {new_name} already exists.")
                else:
                    # Rename the folder
                    os.rename(item_path, new_path)
                    print(f"Renamed {item} to {new_name}")
            else:
                print(f"Skipping {item} because last part '{last_part}' is not a number.")
        else:
            print(f"Skipping {item} because it doesn't have parts after 'json_'.")
    else:
        # Skip non-folder or non-json_ items
        pass

print("Renaming completed.")

Renamed json_3_462 to 462
Renamed json_4_448 to 448
Renamed json_1_611 to 611
Renamed json_1_683 to 683
Renamed json_4_459 to 459
Renamed json_1_552 to 552
Renamed json_4_180 to 180
Renamed json_4_518 to 518
Renamed json_3_1247 to 1247
Renamed json_4_54 to 54
Renamed json_4_186 to 186
Renamed json_3_78 to 78
Renamed json_4_491 to 491
Renamed json_4_166 to 166
Renamed json_4_365 to 365
Renamed json_3_119 to 119
Renamed json_3_1471 to 1471
Renamed json_1_19 to 19
Renamed json_4_246 to 246
Renamed json_1_184 to 184
Renamed json_3_1044 to 1044
Renamed json_3_509 to 509
Renamed json_4_273 to 273
Renamed json_3_1331 to 1331
Renamed json_3_759 to 759
Renamed json_3_84 to 84
Renamed json_4_153 to 153
Renamed json_1_670 to 670
Renamed json_4_148 to 148
Renamed json_3_1738 to 1738
Renamed json_3_1352 to 1352
Renamed json_3_484 to 484
Renamed json_4_570 to 570
Renamed json_3_1614 to 1614
Renamed json_3_1228 to 1228
Renamed json_3_142 to 142
Renamed json_3_1214 to 1214
Renamed json_3_1496 to 1496


In [7]:
# Install faiss before importing
!pip install faiss-cpu  # Use faiss-gpu if you have CUDA and want GPU acceleration

# Now import faiss
import faiss

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 66.6 MB/s eta 0:00:00


In [16]:
!huggingface-cli login


⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: write).
The token `gemma-2-2b-it` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-crede

In [ ]:
# بارگذاری داده‌های آزمون
with open("/content/drive/MyDrive/NLP_HW3_phase3/Question_only_Text.json", "r") as f:
    text_questions = json.load(f)

with open("/content/drive/MyDrive/NLP_HW3_phase3/Question_Text_Image.json", "r") as f:
    multimodal_questions = json.load(f)

# ایجاد سیستم RAG
rag_system = MultimodalRAGSystem(
    clip_model_path="/content/drive/MyDrive/NLP_HW3_phase2/FineTuned_LoRA_CLIP",
    vector_db_path="/content/drive/MyDrive/NLP_HW3_phase2/Persian_Vector_DB",
    gen_model_name="google/gemma-3-1b-it"
)

# ارزیابی مدل پایه روی سوالات متنی
print("ارزیابی مدل پایه روی سوالات متنی...")
base_text_results = evaluate_baseline(text_questions, rag_system.gen_model, rag_system.gen_tokenizer)

# ارزیابی مدل پایه روی سوالات چندوجهی
print("ارزیابی مدل پایه روی سوالات چندوجهی...")
base_multi_results = evaluate_baseline(multimodal_questions, rag_system.gen_model, rag_system.gen_tokenizer, True)

# ارزیابی سیستم RAG روی سوالات متنی
print("ارزیابی سیستم RAG روی سوالات متنی...")
rag_text_results = evaluate_rag_system(text_questions, rag_system)

# ارزیابی سیستم RAG روی سوالات چندوجهی
print("ارزیابی سیستم RAG روی سوالات چندوجهی...")
rag_multi_results = evaluate_rag_system(multimodal_questions, rag_system, True)

# نمایش نتایج
print("\nنتایج نهایی:")
print("=" * 50)
print("مدل پایه - سوالات متنی:")
print(f"دقت: {base_text_results[0]:.4f}, خطای میکرو: {base_text_results[2]:.4f}")

print("مدل پایه - سوالات چندوجهی:")
print(f"دقت: {base_multi_results[0]:.4f}, خطای میکرو: {base_multi_results[2]:.4f}")

print(f"معیارهای بازیابی: HIT@3={rag_multi_results['retrieval_metrics']['HIT@3']:.4f}")
print("سیستم RAG - سوالات متنی:")
print(f"دقت: {rag_text_results['accuracy']:.4f}, خطای میکرو: {rag_text_results['micro_error']:.4f}")

print("سیستم RAG - سوالات چندوجهی:")
print(f"دقت: {rag_multi_results['accuracy']:.4f}, خطای میکرو: {rag_multi_results['micro_error']:.4f}")

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    CLIPVisionModel, CLIPTextModel, CLIPTokenizer, CLIPProcessor,
    AutoModelForCausalLM, AutoTokenizer
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from PIL import Image
from sklearn.model_selection import train_test_split
import json
from tqdm import tqdm
import pickle
from typing import List, Dict, Tuple, Optional
import logging
from dataclasses import dataclass
import warnings
import faiss
warnings.filterwarnings('ignore')

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class DataValidator:
    @staticmethod
    def validate_data_structure(csv_path: str, image_folder: str) -> dict:
        result = {'valid': True, 'errors': [], 'warnings': [], 'stats': {}}
        try:
            data = pd.read_csv(csv_path, dtype={'name': str}) # FIX: Added dtype={'name': str}
            if not all(col in data.columns for col in ['name', 'question', 'answer','iteration']):
                result['errors'].append("Missing required columns in CSV")
                result['valid'] = False
            if not os.path.exists(image_folder):
                result['errors'].append(f"Image folder {image_folder} does not exist")
                result['valid'] = False
            stats = {
                'total_celebrities': len(data),
                'available_celebrities': 0,
                'total_images': 0,
                'avg_images_per_celeb': 0
            }
            image_counts = []
            for _, row in data.iterrows():
                iteration_val = str(row["iteration"]) if pd.notna(row["iteration"]) else ""
                celeb_folder = os.path.join(image_folder, iteration_val.replace(".0", ""))
                if os.path.exists(celeb_folder):
                    stats['available_celebrities'] += 1
                    images = [f for f in os.listdir(celeb_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.webp', '.tiff', '.tif'))]
                    stats['total_images'] += len(images)
                    image_counts.append(len(images))
            if image_counts:
                stats['avg_images_per_celeb'] = sum(image_counts) / len(image_counts)
            else:
                stats['avg_images_per_celeb'] = 0
            result['stats'] = stats
        except Exception as e:
            result['errors'].append(str(e))
            result['valid'] = False
        return result

@dataclass
class TrainingConfig:
    batch_size: int = 8
    learning_rate: float = 2e-5
    epochs: int = 8
    accumulation_steps: int = 4
    max_grad_norm: float = 1.0
    weight_decay: float = 0.01
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    patience: int = 3

class PersianCelebrityDataset(Dataset):
    """Fixed dataset with proper tensor handling"""

    def __init__(self, data_samples: List[Dict], tokenizer, processor, max_length: int = 77):
        self.data_samples = data_samples
        self.tokenizer = tokenizer
        self.processor = processor
        self.max_length = max_length
        self._validate_images()

    def _validate_images(self):
        """Pre-validate all images to avoid runtime errors"""
        valid_samples = []
        for sample in tqdm(self.data_samples, desc="Validating images"):
            if self._is_valid_image(sample['image_path']):
                valid_samples.append(sample)
            else:
                logger.warning(f"Removing invalid image: {sample['image_path']}")
        self.data_samples = valid_samples
        logger.info(f"Dataset validated: {len(self.data_samples)} valid samples")

    def _is_valid_image(self, image_path: str) -> bool:
        """Check if image is valid"""
        try:
            with Image.open(image_path) as img:
                img.verify()
            return True
        except Exception:
            return False

    def safe_load_image(self, image_path: str) -> Image.Image:
        """Robust image loading with fallback"""
        try:
            image = Image.open(image_path).convert('RGB')
            if image.size[0] < 10 or image.size[1] < 10:
                raise ValueError("Image too small")
            return image
        except Exception as e:
            logger.warning(f"Failed to load {image_path}: {e}")
            return Image.new('RGB', (224, 224), color='white')

    def __len__(self):
        return len(self.data_samples)

    def __getitem__(self, idx):
        sample = self.data_samples[idx]
        name = sample['name']
        question = sample['question']
        narrative = sample['answer']
        image_path = sample['image_path']

        text = f"{question} {narrative}"
        image = self.safe_load_image(image_path)

        text_inputs = self.tokenizer(
            text,
            return_tensors="pt",
            padding='max_length',
            truncation=True,
            max_length=self.max_length
        )

        image_inputs = self.processor(
            images=image,
            return_tensors="pt"
        )

        return {
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0),
            'pixel_values': image_inputs['pixel_values'].squeeze(0),
            'text': text,
            'name': name,
            'image_path': image_path
        }

class CLIPWithLoRAFixed(nn.Module):
    """CLIP wrapper with proper dimension handling"""

    def __init__(self, vision_model, text_model):
        super().__init__()
        self.vision_model = vision_model
        self.text_model = text_model

        self.vision_dim = vision_model.config.hidden_size if hasattr(vision_model.config, 'hidden_size') else 768
        self.text_dim = text_model.config.hidden_size if hasattr(text_model.config, 'hidden_size') else 512

        common_dim = 512
        self.vision_projection = nn.Linear(self.vision_dim, common_dim)
        self.text_projection = nn.Linear(self.text_dim, common_dim)

        device = next(vision_model.parameters()).device
        self.vision_projection = self.vision_projection.to(device)
        self.text_projection = self.text_projection.to(device)

        logger.info(f"Initialized CLIPWithLoRAFixed: vision_dim={self.vision_dim}, text_dim={self.text_dim}, common_dim={common_dim}")

    def encode_image(self, pixel_values):
        """Encode images with proper dimension handling"""
        vision_output = self.vision_model(pixel_values=pixel_values)
        image_embeds = vision_output.pooler_output
        image_embeds = self.vision_projection(image_embeds)
        return vision_output, image_embeds

    def encode_text(self, input_ids, attention_mask):
        """Encode text with proper dimension handling"""
        if hasattr(self.text_model, 'base_model'):
            actual_text_model = self.text_model.base_model
            text_output = actual_text_model(input_ids=input_ids, attention_mask=attention_mask)
        else:
            text_output = self.text_model(input_ids=input_ids, attention_mask=attention_mask)

        text_embeds = text_output.pooler_output
        text_embeds = self.text_projection(text_embeds)
        return text_output, text_embeds

def create_fixed_dataloader(dataset, batch_size, shuffle=True, num_workers=0):
    """Create dataloader with custom collation to handle tensor issues"""

    def custom_collate_fn(batch):
        input_ids = []
        attention_masks = []
        pixel_values = []
        texts = []
        names = []
        image_paths = []

        for item in batch:
            input_ids.append(item['input_ids'])
            attention_masks.append(item['attention_mask'])
            pixel_values.append(item['pixel_values'])
            texts.append(item['text'])
            names.append(item['name'])
            image_paths.append(item['image_path'])

        try:
            input_ids = torch.stack(input_ids, dim=0)
            attention_masks = torch.stack(attention_masks, dim=0)
            pixel_values = torch.stack(pixel_values, dim=0)
        except Exception as e:
            logger.error(f"Collation error: {e}")
            return None

        return {
            'input_ids': input_ids,
            'attention_mask': attention_masks,
            'pixel_values': pixel_values,
            'text': texts,
            'name': names,
            'image_path': image_paths
        }

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=custom_collate_fn,
        drop_last=True
    )

class LoRACLIPFineTuner:
    """FIXED: LoRA CLIP Fine-tuner with corrected save/load logic"""

    def __init__(self, model_name: str = "openai/clip-vit-base-patch32", device: str = None, use_mixed_precision: bool = False):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.use_mixed_precision = False  # Force disable
        self.scaler = None

        logger.info(f"Loading CLIP model: {model_name}")
        self._load_models()

        self.temperature = nn.Parameter(torch.tensor(0.07, device=self.device, requires_grad=True))
        self.clip_model = None
        self.lora_applied_to_text_only = True  # TRACK WHICH MODEL HAS LORA

        self.training_stats = {
            'train_losses': [],
            'test_losses': [],
            'learning_rates': [],
            'temperatures': []
        }

        logger.info(f"Loaded CLIP models on {self.device}")

    def _load_models(self):
        """Load CLIP models with error handling"""
        try:
            self.vision_encoder = CLIPVisionModel.from_pretrained(self.model_name).to(self.device)
            self.text_encoder = CLIPTextModel.from_pretrained(self.model_name).to(self.device)
            self.tokenizer = CLIPTokenizer.from_pretrained(self.model_name)
            self.processor = CLIPProcessor.from_pretrained(self.model_name)
        except Exception as e:
            logger.error(f"Failed to load CLIP models: {e}")
            raise

    def apply_lora(self, config):
        """Apply LoRA only to text encoder - FIXED VERSION"""
        logger.info("Applying LoRA to text encoder only...")

        text_lora_config = LoraConfig(
            r=config.lora_r,
            lora_alpha=config.lora_alpha,
            target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
            lora_dropout=config.lora_dropout,
            bias="none",
            task_type=TaskType.FEATURE_EXTRACTION,
        )

        try:
            # Only apply LoRA to text encoder
            self.text_encoder = get_peft_model(self.text_encoder, text_lora_config)
            self.lora_applied_to_text_only = True  # Mark that only text has LoRA

            # Create wrapper model
            self.clip_model = CLIPWithLoRAFixed(
                self.vision_encoder,
                self.text_encoder
            )

            logger.info("LoRA applied successfully to text encoder only")
            self.print_trainable_parameters()

        except Exception as e:
            logger.error(f"Failed to apply LoRA: {e}")
            raise

    def print_trainable_parameters(self):
        """Print trainable parameters info"""
        vision_trainable = sum(p.numel() for p in self.vision_encoder.parameters() if p.requires_grad)
        text_trainable = sum(p.numel() for p in self.text_encoder.parameters() if p.requires_grad)
        vision_total = sum(p.numel() for p in self.vision_encoder.parameters())
        text_total = sum(p.numel() for p in self.text_encoder.parameters())

        logger.info(f"Vision encoder: {vision_trainable:,}/{vision_total:,} trainable parameters ({100*vision_trainable/vision_total:.2f}%)")
        logger.info(f"Text encoder: {text_trainable:,}/{text_total:,} trainable parameters ({100*text_trainable/text_total:.2f}%)")

    def contrastive_loss(self, image_embeddings, text_embeddings):
        """Enhanced contrastive loss"""
        image_embeddings = F.normalize(image_embeddings, dim=1)
        text_embeddings = F.normalize(text_embeddings, dim=1)

        temperature = torch.clamp(self.temperature, min=0.01, max=1.0)
        logits = torch.matmul(image_embeddings, text_embeddings.T) / temperature

        batch_size = logits.shape[0]
        labels = torch.arange(batch_size, device=self.device)

        loss_img = F.cross_entropy(logits, labels, label_smoothing=0.1)
        loss_text = F.cross_entropy(logits.T, labels, label_smoothing=0.1)

        return (loss_img + loss_text) / 2

    def train_epoch(self, dataloader, optimizer, scheduler, epoch, config):
        """FIXED training loop"""
        if self.clip_model is None:
            raise RuntimeError("LoRA not applied! Call apply_lora() first.")

        self.vision_encoder.train()
        self.text_encoder.train()
        total_loss = 0
        num_batches = 0

        optimizer.zero_grad()

        pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
        for batch_idx, batch in enumerate(pbar):
            if batch is None:
                logger.warning(f"Skipping None batch at index {batch_idx}")
                continue

            try:
                input_ids = batch['input_ids'].to(self.device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(self.device, non_blocking=True)
                pixel_values = batch['pixel_values'].to(self.device, non_blocking=True)

                text_output, text_embeds = self.clip_model.encode_text(input_ids, attention_mask)
                image_output, image_embeds = self.clip_model.encode_image(pixel_values)

                loss = self.contrastive_loss(image_embeds, text_embeds)
                loss = loss / config.accumulation_steps

                if torch.isnan(loss) or torch.isinf(loss):
                    logger.warning(f"Invalid loss at batch {batch_idx}: {loss.item()}")
                    continue

                loss.backward()

                if (batch_idx + 1) % config.accumulation_steps == 0:
                    total_norm = torch.nn.utils.clip_grad_norm_(
                        list(self.vision_encoder.parameters()) +
                        list(self.text_encoder.parameters()) +
                        [self.temperature],
                        max_norm=config.max_grad_norm
                    )

                    optimizer.step()
                    optimizer.zero_grad()
                    scheduler.step()

                total_loss += loss.item() * config.accumulation_steps
                num_batches += 1

                current_temp = torch.clamp(self.temperature, min=0.01, max=1.0).item()
                current_lr = scheduler.get_last_lr()[0] if hasattr(scheduler, 'get_last_lr') else config.learning_rate

                pbar.set_postfix({
                    'loss': f'{loss.item() * config.accumulation_steps:.3f}',
                    'temp': f'{current_temp:.3f}',
                    'lr': f'{current_lr:.2e}'
                })

                if batch_idx % 50 == 0:
                    torch.cuda.empty_cache()

            except RuntimeError as e:
                if "out of memory" in str(e):
                    logger.warning(f"CUDA OOM at batch {batch_idx}, skipping batch")
                    torch.cuda.empty_cache()
                    continue
                else:
                    logger.error(f"Training error at batch {batch_idx}: {e}")
                    continue

        return total_loss / max(num_batches, 1)

    def test_epoch(self, dataloader):
        """FIXED test loop"""
        if dataloader is None:
            return float('inf')

        if self.clip_model is None:
            raise RuntimeError("LoRA not applied! Call apply_lora() first.")

        self.vision_encoder.eval()
        self.text_encoder.eval()
        total_loss = 0
        num_batches = 0

        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Testing"):
                if batch is None:
                    continue

                try:
                    input_ids = batch['input_ids'].to(self.device, non_blocking=True)
                    attention_mask = batch['attention_mask'].to(self.device, non_blocking=True)
                    pixel_values = batch['pixel_values'].to(self.device, non_blocking=True)

                    text_output, text_embeds = self.clip_model.encode_text(input_ids, attention_mask)
                    image_output, image_embeds = self.clip_model.encode_image(pixel_values)

                    loss = self.contrastive_loss(image_embeds, text_embeds)

                    if not (torch.isnan(loss) or torch.isinf(loss)):
                        total_loss += loss.item()
                        num_batches += 1

                except RuntimeError as e:
                    if "out of memory" in str(e):
                        logger.warning("CUDA OOM during testing, skipping batch")
                        torch.cuda.empty_cache()
                        continue
                    else:
                        logger.error(f"Test error: {e}")
                        continue

        return total_loss / max(num_batches, 1)

    def create_dataloaders(self, csv_path: str, image_folder: str, config):
        """Create fixed dataloaders"""
        train_samples, test_samples = self.create_data_splits(csv_path, image_folder)

        train_dataset = PersianCelebrityDataset(train_samples, self.tokenizer, self.processor)
        test_dataset = PersianCelebrityDataset(test_samples, self.tokenizer, self.processor)

        train_loader = create_fixed_dataloader(
            train_dataset,
            batch_size=config.batch_size,
            shuffle=True,
            num_workers=0
        )

        test_loader = None
        if len(test_dataset) > 0:
            test_loader = create_fixed_dataloader(
                test_dataset,
                batch_size=config.batch_size,
                shuffle=False,
                num_workers=0
            )

        return train_loader, test_loader

    def create_data_splits(self, csv_path: str, image_folder: str, test_split: float = 0.2):
        """Enhanced data splitting with better validation"""
        validation_result = DataValidator.validate_data_structure(csv_path, image_folder)

        if not validation_result['valid']:
            for error in validation_result['errors']:
                logger.error(error)
            raise ValueError("Data validation failed")

        for warning in validation_result['warnings']:
            logger.warning(warning)

        stats = validation_result['stats']
        logger.info(f"Data validation passed:")
        logger.info(f"  - Total celebrities: {stats['total_celebrities']}")
        logger.info(f"  - Available celebrities: {stats['available_celebrities']}")
        logger.info(f"  - Total images: {stats['total_images']}")
        logger.info(f"  - Avg images per celebrity: {stats['avg_images_per_celeb']:.1f}")

        data = pd.read_csv(csv_path)
        train_samples = []
        test_samples = []

        for _, row in data.iterrows():
            name = row['name']
            iteration_val = str(row["iteration"])
            celebrity_folder = os.path.join(image_folder, iteration_val.replace(".0", ""))
            print(celebrity_folder)
            if not os.path.exists(celebrity_folder):
                continue

            image_files = [f for f in os.listdir(celebrity_folder)
                          if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.webp', '.tiff', '.tif'))]
            print(image_files)
            if len(image_files) < 1:
                continue
            elif len(image_files) == 1:
                train_samples.append({
                    'name': name,
                    'question': row['question'],
                    'answer': row['answer'],
                    'image_path': os.path.join(celebrity_folder, image_files[0])
                })
            else:
                image_files.sort()
                n_test = max(1, int(len(image_files) * test_split))
                test_images = image_files[-n_test:]
                train_images = image_files[:-n_test]

                for img_file in train_images:
                    train_samples.append({
                        'name': name,
                        'question': row['question'],
                        'answer': row['answer'],
                        'image_path': os.path.join(celebrity_folder, img_file)
                    })

                for img_file in test_images:
                    test_samples.append({
                        'name': name,
                        'question': row['question'],
                        'answer': row['answer'],
                        'image_path': os.path.join(celebrity_folder, img_file)
                    })

        logger.info(f"Created {len(train_samples)} training samples and {len(test_samples)} test samples")

        if len(test_samples) == 0:
            logger.warning("No test samples created. Consider adding more images per celebrity.")

        return train_samples, test_samples

    def save_checkpoint(self, save_path: str, epoch: int, train_loss: float, test_loss: float, is_best: bool = False):
        """Save training checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'train_loss': train_loss,
            'test_loss': test_loss,
            'temperature': self.temperature.data.item(),
            'training_stats': self.training_stats
        }

        checkpoint_path = os.path.join(save_path, f"checkpoint_epoch_{epoch}.pt")
        torch.save(checkpoint, checkpoint_path)

        if is_best:
            best_path = os.path.join(save_path, "best_checkpoint.pt")
            torch.save(checkpoint, best_path)
            logger.info(f"Saved best checkpoint at epoch {epoch}")

    def fine_tune(self, csv_path: str, image_folder: str, save_path: str = "finetuned_lora_clip"):
        """Complete fine-tuning pipeline with LoRA"""
        os.makedirs(save_path, exist_ok=True)

        config = TrainingConfig()
        logger.info(f"Training configuration:")
        logger.info(f"  - Batch Size: {config.batch_size}")
        logger.info(f"  - Learning Rate: {config.learning_rate:.2e}")
        logger.info(f"  - Accumulation Steps: {config.accumulation_steps}")
        logger.info(f"  - LoRA r: {config.lora_r}")
        logger.info(f"  - LoRA alpha: {config.lora_alpha}")

        # Apply LoRA
        self.apply_lora(config)

        # Create dataloaders
        logger.info("Creating dataloaders...")
        train_loader, test_loader = self.create_dataloaders(csv_path, image_folder, config)

        # Setup optimizer - INCLUDE PROJECTION LAYER
        trainable_params = (
            [p for p in self.vision_encoder.parameters() if p.requires_grad] +
            [p for p in self.text_encoder.parameters() if p.requires_grad] +
            list(self.clip_model.vision_projection.parameters()) +
            list(self.clip_model.text_projection.parameters()) +
            [self.temperature]
        )
        optimizer = AdamW(trainable_params, lr=config.learning_rate, weight_decay=config.weight_decay)

        # Learning rate scheduler
        total_steps = len(train_loader) * config.epochs // config.accumulation_steps
        scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=config.learning_rate/10)

        # Training loop
        best_test_loss = float('inf')
        patience_counter = 0

        for epoch in range(config.epochs):
            logger.info(f"\nEpoch {epoch+1}/{config.epochs}")

            # Train
            train_loss = self.train_epoch(train_loader, optimizer, scheduler, epoch+1, config)

            # Test
            test_loss = self.test_epoch(test_loader)

            # Update training stats
            current_temp = torch.clamp(self.temperature, min=0.01, max=1.0).item()
            current_lr = scheduler.get_last_lr()[0] if hasattr(scheduler, 'get_last_lr') else config.learning_rate

            self.training_stats['train_losses'].append(train_loss)
            self.training_stats['test_losses'].append(test_loss)
            self.training_stats['learning_rates'].append(current_lr)
            self.training_stats['temperatures'].append(current_temp)

            logger.info(f"Results:")
            logger.info(f"  - Train Loss: {train_loss:.4f}")
            logger.info(f"  - Test Loss: {test_loss:.4f}")
            logger.info(f"  - Temperature: {current_temp:.4f}")
            logger.info(f"  - Learning Rate: {current_lr:.2e}")

            # Save checkpoint
            is_best = test_loss < best_test_loss
            self.save_checkpoint(save_path, epoch+1, train_loss, test_loss, is_best)

            # Early stopping
            if is_best:
                best_test_loss = test_loss
                patience_counter = 0
                self.save_model(save_path)
                logger.info(f"New best model! Test loss: {test_loss:.4f}")
            else:
                patience_counter += 1
                logger.info(f"No improvement. Patience: {patience_counter}/{config.patience}")

                if patience_counter >= config.patience:
                    logger.info("Early stopping triggered!")
                    break

        logger.info(f"\nTraining completed!")
        logger.info(f"Best test loss: {best_test_loss:.4f}")

        # Save final training stats
        stats_path = os.path.join(save_path, "training_stats.json")
        with open(stats_path, 'w') as f:
            json.dump(self.training_stats, f, indent=2)

    def save_model(self, save_path: str):
        """FIXED: Enhanced model saving for LoRA - only save what was trained"""
        os.makedirs(save_path, exist_ok=True)

        try:
            # Save LoRA adapters (ONLY TEXT ENCODER HAS LORA)
            if self.lora_applied_to_text_only:
                self.text_encoder.save_pretrained(os.path.join(save_path, "text_lora"))
                logger.info("Saved text LoRA adapters")

            # Save vision encoder state dict (no LoRA)
            torch.save(self.vision_encoder.state_dict(), os.path.join(save_path, "vision_encoder.pt"))
            logger.info("Saved vision encoder state dict")

            # Save tokenizer and processor
            self.tokenizer.save_pretrained(os.path.join(save_path, "tokenizer"))
            self.processor.save_pretrained(os.path.join(save_path, "processor"))

            # Save projection layers
            projection_state = {
                'vision_projection': self.clip_model.vision_projection.state_dict(),
                'text_projection': self.clip_model.text_projection.state_dict(),
            }
            torch.save(projection_state, os.path.join(save_path, "projection_layers.pt"))

            # Save additional parameters
            torch.save({
                'temperature': self.temperature.data.item(),
                'training_stats': self.training_stats,
                'base_model_name': self.model_name,
                'lora_applied_to_text_only': self.lora_applied_to_text_only
            }, os.path.join(save_path, "additional_params.pt"))

            # Save model configuration
            config_data = {
                'model_type': 'LoRA_CLIP',
                'base_model': self.model_name,
                'temperature': self.temperature.data.item(),
                'device': str(self.device),
                'mixed_precision': self.use_mixed_precision,
                'lora_applied_to_text_only': self.lora_applied_to_text_only
            }

            with open(os.path.join(save_path, "config.json"), 'w') as f:
                json.dump(config_data, f, indent=2)

            logger.info(f"LoRA CLIP model saved to {save_path}")

        except Exception as e:
            logger.error(f"Failed to save model: {e}")
            raise

    def load_model(self, model_path: str):
        """FIXED: Enhanced model loading for LoRA - only load what was saved"""
        try:
            # Load additional parameters first to get base model name
            checkpoint = torch.load(os.path.join(model_path, "additional_params.pt"), map_location=self.device)
            base_model_name = checkpoint.get('base_model_name', self.model_name)
            self.lora_applied_to_text_only = checkpoint.get('lora_applied_to_text_only', True)

            # Load base models first
            self.vision_encoder = CLIPVisionModel.from_pretrained(base_model_name).to(self.device)
            self.text_encoder = CLIPTextModel.from_pretrained(base_model_name).to(self.device)

            # Load vision encoder weights (no LoRA applied)
            vision_state_path = os.path.join(model_path, "vision_encoder.pt")
            if os.path.exists(vision_state_path):
                vision_state = torch.load(vision_state_path, map_location=self.device)
                self.vision_encoder.load_state_dict(vision_state)
                logger.info("Loaded vision encoder weights")

            # Load LoRA adapters ONLY for text encoder
            if self.lora_applied_to_text_only:
                text_lora_path = os.path.join(model_path, "text_lora")
                if os.path.exists(text_lora_path):
                    self.text_encoder = PeftModel.from_pretrained(
                        self.text_encoder,
                        text_lora_path
                    )
                    logger.info("Loaded text LoRA adapters")
                else:
                    logger.warning("Text LoRA path not found, using base text encoder")

            # Load tokenizer and processor
            self.tokenizer = CLIPTokenizer.from_pretrained(
                os.path.join(model_path, "tokenizer")
            )
            self.processor = CLIPProcessor.from_pretrained(
                os.path.join(model_path, "processor")
            )

            # Create wrapper model
            self.clip_model = CLIPWithLoRAFixed(self.vision_encoder, self.text_encoder)

            # Load projection layers
            projection_path = os.path.join(model_path, "projection_layers.pt")
            if os.path.exists(projection_path):
                projection_state = torch.load(projection_path, map_location=self.device)
                self.clip_model.vision_projection.load_state_dict(projection_state['vision_projection'])
                self.clip_model.text_projection.load_state_dict(projection_state['text_projection'])
                logger.info("Loaded projection layers")

            # Create new temperature parameter
            self.temperature = nn.Parameter(
                torch.tensor(checkpoint['temperature'], device=self.device, requires_grad=True)
            )

            if 'training_stats' in checkpoint:
                self.training_stats = checkpoint['training_stats']

            logger.info(f"LoRA CLIP model loaded from {model_path}")

        except Exception as e:
            logger.error(f"Failed to load model: {e}")
            raise


class PersianMultimodalEmbeddingExtractor:
    """Extract embeddings from fine-tuned LoRA CLIP model"""

    def __init__(self, model_path: str, device: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model_path = model_path

        # Load the fine-tuned LoRA model
        self.fine_tuner = LoRACLIPFineTuner(device=self.device, use_mixed_precision=False)
        self.fine_tuner.load_model(model_path)

        # Set to evaluation mode
        self.fine_tuner.vision_encoder.eval()
        self.fine_tuner.text_encoder.eval()

        logger.info(f"Embedding extractor ready on {self.device}")

    def extract_text_embeddings(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """Extract embeddings from text"""
        all_embeddings = []

        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch_texts = texts[i:i+batch_size]

                # Tokenize batch
                inputs = self.fine_tuner.tokenizer(
                    batch_texts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=77  # CLIP max length
                ).to(self.device)

                # Get embeddings using the wrapper
                _, embeddings = self.fine_tuner.clip_model.encode_text(
                    inputs['input_ids'],
                    inputs['attention_mask']
                )

                # Normalize embeddings
                embeddings = F.normalize(embeddings, dim=1)

                all_embeddings.append(embeddings.cpu().numpy())

        return np.vstack(all_embeddings)

    def extract_image_embeddings(self, image_paths: List[str], batch_size: int = 32) -> np.ndarray:
        """Extract embeddings from images"""
        all_embeddings = []

        with torch.no_grad():
            for i in range(0, len(image_paths), batch_size):
                batch_paths = image_paths[i:i+batch_size]
                batch_images = []

                # Load and preprocess images
                for path in batch_paths:
                    try:
                        image = Image.open(path).convert('RGB')
                        batch_images.append(image)
                    except Exception as e:
                        logger.warning(f"Failed to load {path}: {e}")
                        # Use dummy image
                        batch_images.append(Image.new('RGB', (224, 224), color='white'))

                # Preprocess batch
                inputs = self.fine_tuner.processor(
                    images=batch_images,
                    return_tensors="pt"
                ).to(self.device)

                # Get embeddings using the wrapper
                _, embeddings = self.fine_tuner.clip_model.encode_image(inputs['pixel_values'])

                # Normalize embeddings
                embeddings = F.normalize(embeddings, dim=1)

                all_embeddings.append(embeddings.cpu().numpy())

        return np.vstack(all_embeddings)

    def compute_similarity(self, text_embeddings: np.ndarray, image_embeddings: np.ndarray) -> np.ndarray:
        """Compute cosine similarity between text and image embeddings"""
        # Ensure embeddings are normalized
        text_embeddings = text_embeddings / np.linalg.norm(text_embeddings, axis=1, keepdims=True)
        image_embeddings = image_embeddings / np.linalg.norm(image_embeddings, axis=1, keepdims=True)

        return np.dot(text_embeddings, image_embeddings.T)


class PersianVectorDatabase:
    """Vector database for Persian celebrity embeddings"""

    def __init__(self, db_type: str = "faiss", db_path: str = "persian_vector_db"):
        self.db_type = db_type
        self.db_path = db_path
        self.embedding_dim = 512  # CLIP embedding dimension

        if db_type == "faiss":
            self._init_faiss()
        elif db_type == "chromadb":
            self._init_chromadb()
        else:
            raise ValueError(f"Unsupported database type: {db_type}")

        self.metadata = []  # Store celebrity info
        self.is_built = False

    def _init_faiss(self):
        """Initialize FAISS index"""
        self.index = faiss.IndexFlatIP(self.embedding_dim)  # Inner product for cosine similarity

    def _init_chromadb(self):
        """Initialize ChromaDB"""
        try:
            import chromadb
        except ImportError:
            raise ImportError("ChromaDB not installed. Install with pip install chromadb")
        self.client = chromadb.PersistentClient(path=self.db_path)
        self.collection = self.client.get_or_create_collection(
            name="persian_celebrities",
            metadata={"hnsw:space": "cosine"}
        )

    def add_embeddings(self, embeddings: np.ndarray, metadata: List[Dict]):
        """Add embeddings to database"""
        if self.db_type == "faiss":
            # Normalize for cosine similarity
            normalized_embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
            self.index.add(normalized_embeddings.astype(np.float32))
            self.metadata.extend(metadata)

        elif self.db_type == "chromadb":
            # ChromaDB expects lists
            embeddings_list = embeddings.tolist()
            ids = [f"celeb_{i}" for i in range(len(embeddings))]
            documents = [meta.get('description', '') for meta in metadata]
            metadatas = metadata

            self.collection.add(
                embeddings=embeddings_list,
                ids=ids,
                documents=documents,
                metadatas=metadatas
            )

        self.is_built = True
        logger.info(f"Added {len(embeddings)} embeddings to {self.db_type} database")

    def search(self, query_embedding: np.ndarray, top_k: int = 5) -> List[Dict]:
        """Search for similar embeddings"""
        if not self.is_built:
            raise RuntimeError("Database not built yet. Add embeddings first.")

        query_embedding = query_embedding.reshape(1, -1)

        if self.db_type == "faiss":
            # Normalize query
            query_embedding = query_embedding / np.linalg.norm(query_embedding)
            scores, indices = self.index.search(query_embedding.astype(np.float32), top_k)

            results = []
            for score, idx in zip(scores[0], indices[0]):
                if idx < len(self.metadata):
                    result = self.metadata[idx].copy()
                    result['similarity'] = float(score)
                    results.append(result)

            return results

        elif self.db_type == "chromadb":
            results = self.collection.query(
                query_embeddings=query_embedding.tolist(),
                n_results=top_k
            )

            formatted_results = []
            for i in range(len(results['ids'][0])):
                result = {
                    'id': results['ids'][0][i],
                    'similarity': 1 - results['distances'][0][i],  # Convert distance to similarity
                    'metadata': results['metadatas'][0][i],
                    'document': results['documents'][0][i]
                }
                formatted_results.append(result)

            return formatted_results

    def save(self, path: str = None):
        """Save database to disk"""
        save_path = path or self.db_path
        os.makedirs(save_path, exist_ok=True)

        if self.db_type == "faiss":
            faiss.write_index(self.index, os.path.join(save_path, "index.faiss"))
            with open(os.path.join(save_path, "metadata.pkl"), 'wb') as f:
                pickle.dump(self.metadata, f)
        # ChromaDB is already persistent

        logger.info(f"Database saved to {save_path}")

    def load(self, path: str = None):
        """Load database from disk"""
        load_path = path or self.db_path

        if self.db_type == "faiss":
            index_path = os.path.join(load_path, "index.faiss")
            metadata_path = os.path.join(load_path, "metadata.pkl")
            if os.path.exists(index_path) and os.path.exists(metadata_path):
                self.index = faiss.read_index(index_path)
                with open(metadata_path, 'rb') as f:
                    self.metadata = pickle.load(f)
                self.is_built = True
                logger.info(f"FAISS database loaded from {load_path}")
            else:
                logger.warning(f"No saved FAISS index or metadata found at {load_path}")
        # ChromaDB loads automatically from persistent path


class Gemma3VisionLanguageModel:
    """Fixed Gemma 3 Vision-Language Model for Persian Celebrity RAG"""

    def __init__(self, model_name: str = "google/gemma-2-2b-it", device: str = None):
        self.model_name = model_name
        torch._dynamo.config.disable = True
        torch.backends.cuda.enable_math_sdp(False)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        logger.info(f"Initializing Gemma 3 model: {model_name}")
        self._load_model()

    def _load_model(self):
        """Load Gemma 3 vision-language model"""
        try:
            # For now, using Gemma 2 as Gemma 3 multimodal may not be widely available yet
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                torch_dtype=torch.float16,
                device_map="auto",
                trust_remote_code=True,
                attn_implementation="eager"

            )

            self.tokenizer = AutoTokenizer.from_pretrained(
                self.model_name,
                trust_remote_code=True
            )

            # Ensure pad token is set
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            logger.info(f"Successfully loaded Gemma model: {self.model_name}")

        except Exception as e:
            logger.error(f"Failed to load Gemma model: {e}")
            # Fallback to a simpler model
            self._load_fallback_model()

    def _load_fallback_model(self):
        """Load fallback Gemma model without multimodal features"""
        try:
            fallback_model = "google/gemma-2-2b-it"
            logger.info(f"Loading fallback model: {fallback_model}")

            self.model = AutoModelForCausalLM.from_pretrained(
                fallback_model,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                trust_remote_code=True
            )

            self.tokenizer = AutoTokenizer.from_pretrained(fallback_model)

            # Ensure pad token is set
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            self.model_name = fallback_model

            logger.info(f"Successfully loaded fallback model: {fallback_model}")

        except Exception as e:
            logger.error(f"Failed to load fallback model: {e}")
            raise RuntimeError("Unable to load any Gemma model")

    def generate_response(self, prompt: str, image: Image.Image = None, **kwargs) -> str:
        """Generate response using Gemma model - FIXED VERSION"""
        try:
            # Tokenize the prompt
            inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=2048
            ).to(self.device)

            input_ids = inputs.input_ids
            attention_mask = inputs.attention_mask

            # Get the length of input for proper slicing
            input_length = input_ids.shape[1]

            # Generate response
            with torch.no_grad():
                outputs = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=kwargs.get('max_tokens', 500),
                    temperature=kwargs.get('temperature', 0.7),
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    repetition_penalty=1.1,
                    top_p=0.9
                )

            # Decode only the new tokens (response part)
            # Fix: Properly handle tensor slicing
            if len(outputs.shape) == 2 and outputs.shape[0] == 1:
                # Single batch case
                response_tokens = outputs[0, input_length:]
            else:
                # Multi-batch case or different tensor structure
                response_tokens = outputs[:, input_length:]
                if response_tokens.dim() == 2:
                    response_tokens = response_tokens[0]

            # Decode the response
            response = self.tokenizer.decode(response_tokens, skip_special_tokens=True)

            # Clean up the response
            response = response.strip()

            # If response is empty or too short, provide a default
            if len(response.strip()) < 10:
                return "متأسفم، نمی‌توانم پاسخ مناسبی در این لحظه ارائه دهم. لطفا سوال خود را دوباره مطرح کنید."

            return response

        except Exception as e:
            logger.error(f"Failed to generate response: {e}")
            import traceback
            logger.error(f"Traceback: {traceback.format_exc()}")
            return "متأسفم، نمی‌توانم در این لحظه پاسخ مناسبی تولید کنم."

    def generate_persian_response(self, context: str, query: str, **kwargs) -> str:
        """Generate Persian response with better prompting"""

        # Create a more structured Persian prompt
        system_prompt = """شما یک دستیار هوشمند هستید که درباره چهره‌های مشهور ایرانی اطلاعات می‌دهد. پاسخ‌های خود را به زبان فارسی و با جزئیات کافی ارائه دهید."""

        user_prompt = f"""بر اساس اطلاعات زیر، به سوال پاسخ دهید:

          اطلاعات موجود:
          {context}

          سوال: {query}

          پاسخ:"""

        full_prompt = f"{system_prompt}\n\n{user_prompt}"

        return self.generate_response(
            prompt=full_prompt,
            max_tokens=kwargs.get('max_tokens', 400),
            temperature=kwargs.get('temperature', 0.7)
        )


# Updated PersianCelebrityRAGSystem class
class PersianCelebrityRAGSystem:
    """Complete RAG system with LoRA CLIP retrieval and Gemma 3 generation"""

    def __init__(self, clip_model_path: str, vector_db_path: str,
                 gemma_model_name: str = "google/gemma-2-2b-it"):

        self.clip_model_path = clip_model_path
        self.vector_db_path = vector_db_path
        self.gemma_model_name = gemma_model_name

        # Initialize components
        self._initialize_system()

    def _initialize_system(self):
        """Initialize all system components"""
        try:
            # Load CLIP retrieval components
            logger.info("Loading LoRA CLIP embedding extractor...")
            self.embedding_extractor = PersianMultimodalEmbeddingExtractor(self.clip_model_path)

            # Load vector database
            logger.info("Loading vector database...")
            self.vector_db = PersianVectorDatabase(db_type="faiss", db_path=self.vector_db_path)
            self.vector_db.load()

            # Load Gemma 3 model
            logger.info("Loading Gemma 3 vision-language model...")
            self.gemma_model = Gemma3VisionLanguageModel(self.gemma_model_name)

            logger.info("Persian Celebrity RAG System with LoRA CLIP initialized successfully!")

        except Exception as e:
            logger.error(f"Failed to initialize system: {e}")
            raise

    def search_celebrities(self, query: str, top_k: int = 5) -> List[Dict]:
        """Search for celebrities using text query"""
        # Extract query embedding
        query_embedding = self.embedding_extractor.extract_text_embeddings([query])

        # Search database
        results = self.vector_db.search(query_embedding[0], top_k=top_k)

        return results

    def generate_celebrity_response(self, query: str, search_results: List[Dict],
                                  context_image: Image.Image = None) -> str:
        """Generate response about celebrities using Gemma 3"""

        # Create context from search results
        context_parts = []
        for i, result in enumerate(search_results[:3], 1):  # Use top 3 results
            if 'metadata' in result:
                metadata = result['metadata']
            else:
                metadata = {k: v for k, v in result.items() if k != 'similarity'}

            name = metadata.get('name', 'نامشخص')
            description = metadata.get('description', metadata.get('question', ''))
            similarity = result.get('similarity', 0)

            context_parts.append(f"{i}. {name} (مشابهت: {similarity:.3f})\n   توضیحات: {description}")

        context = "\n\n".join(context_parts)

        # Use the improved Persian response generation
        response = self.gemma_model.generate_persian_response(
            context=context,
            query=query,
            max_tokens=400,
            temperature=0.7
        )

        return response

    def process_query(self, query: str, image: Image.Image = None, top_k: int = 5) -> Dict:
        """Complete RAG pipeline: search + generate"""

        # Search for relevant celebrities
        search_results = self.search_celebrities(query, top_k=top_k)
        print("search_results from database :" , search_results)
        # Generate response
        response = self.generate_celebrity_response(query, search_results, image)

        return {
            'query': query,
            'search_results': search_results,
            'response': response,
            'total_results': len(search_results)
        }

def build_persian_celebrity_database(csv_path: str, image_folder: str, model_path: str,
                                   db_type: str = "faiss", db_path: str = "persian_vector_db"):
    """Build complete vector database from LoRA fine-tuned model"""

    logger.info("Building Persian celebrity vector database with LoRA CLIP...")

    # Initialize embedding extractor
    extractor = PersianMultimodalEmbeddingExtractor(model_path)

    # Initialize vector database
    vector_db = PersianVectorDatabase(db_type=db_type, db_path=db_path)

    # Load celebrity data, ensuring 'name' and 'iteration' are strings
    data = pd.read_csv(csv_path, dtype={'name': str, 'iteration': str})

    all_text_embeddings = []
    all_image_embeddings = []
    all_metadata = []

    logger.info("Processing celebrities...")
    for _, row in tqdm(data.iterrows(), total=len(data), desc="Processing celebrities"):
        name = row['name']
        question = row['question']
        narrative = row['answer']
        iteration = row['iteration'] # Use the iteration column for folder mapping

        # Use iteration to find the folder, fall back to name if needed
        celebrity_folder = os.path.join(image_folder, iteration)
        if not os.path.exists(celebrity_folder):
            # Fallback for older data structure if iteration folder doesn't exist
            celebrity_folder = os.path.join(image_folder, iteration)
            if not os.path.exists(celebrity_folder):
                continue

        # Get all images for this celebrity
        image_files = [f for f in os.listdir(celebrity_folder)
                      if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.webp', '.tiff', '.tif'))]

        if not image_files:
            continue

        # Combine text information
        combined_text = f"{question} {narrative}"

        # Extract text embeddings
        text_embeddings = extractor.extract_text_embeddings([combined_text])

        # Extract image embeddings
        image_paths = [os.path.join(celebrity_folder, img) for img in image_files]
        image_embeddings = extractor.extract_image_embeddings(image_paths)

        # Create metadata for each image
        for i, img_file in enumerate(image_files):
            metadata = {
                'name': name,
                'question': question,
                'answer': narrative,
                'image_file': img_file,
                'image_path': os.path.join(celebrity_folder, img_file),
                'description': combined_text
            }
            all_metadata.append(metadata)

        all_text_embeddings.append(np.repeat(text_embeddings, len(image_files), axis=0))
        all_image_embeddings.append(image_embeddings)

    if not all_text_embeddings:
        raise ValueError("No valid celebrity data found!")

    # Combine all embeddings
    final_text_embeddings = np.vstack(all_text_embeddings)
    final_image_embeddings = np.vstack(all_image_embeddings)

    # Combine text and image embeddings (you can experiment with different fusion methods)
    combined_embeddings = (final_text_embeddings + final_image_embeddings) / 2

    # Add to database
    vector_db.add_embeddings(combined_embeddings, all_metadata)

    # Save database
    vector_db.save()

    logger.info(f"Database built successfully!")
    logger.info(f"   - Total embeddings: {len(combined_embeddings)}")
    logger.info(f"   - Celebrities processed: {len(set(meta['name'] for meta in all_metadata))}")
    logger.info(f"   - Database type: {db_type}")
    logger.info(f"   - Saved to: {db_path}")

    return vector_db, extractor




def create_demo_system(clip_model_path: str, vector_db_path: str) -> PersianCelebrityRAGSystem:
    """Create a demo RAG system with LoRA CLIP"""
    return PersianCelebrityRAGSystem(
        clip_model_path=clip_model_path,
        vector_db_path=vector_db_path,
        gemma_model_name="google/gemma-2-2b-it"  # Will be updated to Gemma 3 when available
    )


# Main execution pipeline
if __name__ == "__main__":
    import os

    # ===== Paths & config =====
    CSV_PATH = "/content/questions_answers_shuffled.csv"
    IMAGE_FOLDER = "/content/hw3/all_image_folders2"
    MODEL_SAVE_PATH = "/content/drive/MyDrive/nlp3_1/FineTuned_LoRA_CLIP"
    DB_TYPE = "faiss"  # or "chromadb"
    DB_PATH = "/content/drive/MyDrive/nlp3_1/Persian_Vector_DB"

    def _assert_path(p):
        if not os.path.exists(p):
            raise FileNotFoundError(f"Path not found: {p}")

    def _faiss_index_exists(p):
        return os.path.exists(os.path.join(p, "index.faiss"))

    try:
        # --- sanity checks (esp. important on Colab/Drive) ---
        _assert_path(os.path.dirname(CSV_PATH))
        _assert_path(CSV_PATH)
        _assert_path(IMAGE_FOLDER)
        os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
        os.makedirs(DB_PATH, exist_ok=True)

        # --- Train LoRA CLIP model if not exists ---
        if not os.path.exists(os.path.join(MODEL_SAVE_PATH, "config.json")):
            logger.info("No trained model found — starting LoRA CLIP fine-tuning...")

            # Initialize fine-tuner
            fine_tuner = LoRACLIPFineTuner(
                model_name="openai/clip-vit-base-patch32",
                device="cuda" if torch.cuda.is_available() else "cpu"
            )

            # Fine-tune the model
            fine_tuner.fine_tune(
                csv_path=CSV_PATH,
                image_folder=IMAGE_FOLDER,
                save_path=MODEL_SAVE_PATH
            )

            logger.info("LoRA CLIP fine-tuning completed!")
        else:
            logger.info("Existing LoRA CLIP model found — will be loaded by the RAG system.")

        # --- build the vector DB if needed (load-or-build) ---
        if DB_TYPE == "faiss" and not _faiss_index_exists(DB_PATH):
            logger.info("No FAISS index found — building vector DB now...")
            # Uses the fine-tuned LoRA CLIP saved in MODEL_SAVE_PATH
            vector_db, extractor = build_persian_celebrity_database(
                csv_path=CSV_PATH,
                image_folder=IMAGE_FOLDER,
                model_path=MODEL_SAVE_PATH,
                db_type=DB_TYPE,
                db_path=DB_PATH
            )
            logger.info("Vector DB built and saved.")
        else:
            logger.info("Existing vector DB detected — will be loaded by the RAG system.")

        # --- now create the RAG system (this should load the DB/model) ---
        logger.info("Creating Persian Celebrity RAG System with LoRA CLIP...")
        rag_system = create_demo_system(MODEL_SAVE_PATH, DB_PATH)

        # --- quick single query test ---
        query = "تصویری از حسام‌الدین آشنا نشان بده"
        result = rag_system.process_query(query)
        print(result["response"])

        # --- batch tests ---
        logger.info("Testing RAG system with multiple queries...")
        test_queries = [
            "کامبیز آتابای چه سمت‌هایی در دربار پهلوی داشته است؟	",
        ]
        for q in test_queries:
            out = rag_system.process_query(q)
            print(f"\nسوال: {q}")
            print(f"پاسخ: {out['response']}")
            print("-" * 60)

        logger.info("PERSIAN CELEBRITY RAG SYSTEM WITH LORA CLIP + GEMMA 3 READY!")

    except Exception as e:
        logger.error(f"System setup failed: {e}")
        import traceback
        traceback.print_exc()
        raise


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

search_results from database : [{'name': 'بورا نوگیرا', 'question': 'بورا نوگیرا در کدام تیم به صورت قرضی بازی کرده است؟', 'answer': 'او به صورت قرضی در آوش از سال ۲۰۱۸ تا ۲۰۱۹ بازی کرده است.', 'image_file': '001.jpeg', 'image_path': '/content/hw3/all_image_folders2/504/001.jpeg', 'description': 'بورا نوگیرا در کدام تیم به صورت قرضی بازی کرده است؟ او به صورت قرضی در آوش از سال ۲۰۱۸ تا ۲۰۱۹ بازی کرده است.', 'similarity': 0.6774832010269165}, {'name': 'بوژیدار رادوشویچ', 'question': 'بوژیدار رادوشویچ در چه کشوری متولد شده است؟', 'answer': 'او در یوگسلاوی متولد شده است.', 'image_file': '003.jpeg', 'image_path': '/content/hw3/all_image_folders2/224/003.jpeg', 'description': 'بوژیدار رادوشویچ در چه کشوری متولد شده است؟ او در یوگسلاوی متولد شده است.', 'similarity': 0.6772146224975586}, {'name': 'بورا نوگیرا', 'question': 'بورا نوگیرا در کدام سال\u200cها برای تیم فارنسی بازی کرده است؟', 'answer': 'او از سال ۲۰۱۹ تا ۲۰۲۲ برای فارنسی بازی کرده است.', 'image_file': '001.jpeg', 'image_path': '/co